# CIFAR-10 Demo: Transform Visualization + Baseline vs Augmented Accuracy
This notebook combines both demos:
1. **Visualization of transformed images**
2. **Training baseline vs augmented models**


## 1. Imports

In [ ]:
import torch
import torchvision
import torchvision.transforms as transforms
from torchvision.utils import make_grid
import matplotlib.pyplot as plt
import numpy as np

import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader

torch.manual_seed(42)

## 2. Define Transformation Pipelines

In [ ]:
# Training pipeline (preprocessing → augmentation → normalization)
train_transforms = transforms.Compose([
    transforms.Resize((32, 32)),
    transforms.RandomCrop(32, padding=4),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

# Baseline (no augmentation)
baseline_transforms = transforms.Compose([
    transforms.Resize((32,32)),
    transforms.ToTensor(),
    transforms.Normalize((0.4914,0.4822,0.4465),(0.2470,0.2435,0.2616))
])

# Test pipeline
test_transforms = baseline_transforms

## 3. Load CIFAR-10 Dataset

In [ ]:
train_aug_dataset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=train_transforms)
train_base_dataset = torchvision.datasets.CIFAR10('./data', train=True, download=True, transform=baseline_transforms)
test_dataset       = torchvision.datasets.CIFAR10('./data', train=False, download=True, transform=test_transforms)

train_aug_loader = DataLoader(train_aug_dataset, batch_size=32, shuffle=True)
train_base_loader = DataLoader(train_base_dataset, batch_size=32, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

## 4. Visualization of Augmented Samples

In [ ]:
def imshow(img):
    img = img / 2 + 0.5  # approximate unnormalize
    npimg = img.numpy().transpose((1,2,0))
    plt.figure(figsize=(6,6))
    plt.imshow(npimg)
    plt.axis('off')

dataiter = iter(train_aug_loader)
images, labels = next(dataiter)

imshow(make_grid(images[:16], nrow=4))
plt.show()

## 5. Define a Simple CNN

In [ ]:
class SimpleCNN(nn.Module):
    def __init__(self):
        super().__init__()
        self.conv1 = nn.Conv2d(3, 32, 3, padding=1)
        self.conv2 = nn.Conv2d(32, 64, 3, padding=1)
        self.pool = nn.MaxPool2d(2, 2)
        self.fc1 = nn.Linear(64 * 8 * 8, 256)
        self.fc2 = nn.Linear(256, 10)

    def forward(self, x):
        x = self.pool(F.relu(self.conv1(x)))
        x = self.pool(F.relu(self.conv2(x)))
        x = x.view(-1, 64*8*8)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

## 6. Training and Evaluation Functions

In [ ]:
def train(model, loader, criterion, optimizer, epochs=2):
    model.train()
    for epoch in range(epochs):
        total_loss = 0
        for images, labels in loader:
            outputs = model(images)
            loss = criterion(outputs, labels)
            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()
        print(f"Epoch {epoch+1}, Loss: {total_loss/len(loader):.4f}")

def accuracy(model, loader):
    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for images, labels in loader:
            outputs = model(images)
            _, preds = torch.max(outputs, 1)
            correct += (preds == labels).sum().item()
            total += labels.size(0)
    return correct / total

## 7. Train Baseline Model (No Augmentation)

In [ ]:
baseline_model = SimpleCNN()
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(baseline_model.parameters(), lr=0.001)

print("Training baseline model...")
train(baseline_model, train_base_loader, criterion, optimizer)

base_acc = accuracy(baseline_model, test_loader)
print(f"Baseline Accuracy: {base_acc*100:.2f}%")

## 8. Train Augmented Model

In [ ]:
aug_model = SimpleCNN()
optimizer = optim.Adam(aug_model.parameters(), lr=0.001)

print("Training augmented model...")
train(aug_model, train_aug_loader, criterion, optimizer)

aug_acc = accuracy(aug_model, test_loader)
print(f"Augmented Accuracy: {aug_acc*100:.2f}%")

## 9. Summary
Expected outcome:
- Augmented accuracy > Baseline accuracy
